# Exploración de Online Retail II

Este cuaderno documenta el perfilado que originó las reglas de limpieza del pipeline.
No produce el dashboard: produce las **decisiones** que el dashboard aplica, y deja
por escrito por qué cada una está justificada.

Fuente: transacciones reales de un minorista online británico de artículos de regalo
(1 dic 2009 – 9 dic 2011), publicadas por el UCI Machine Learning Repository.

> Requiere haber ejecutado antes `python src/ingest.py`.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("CREATE VIEW r AS SELECT * FROM read_parquet('../data/raw/online_retail_II.parquet')")

def q(sql):
    return con.execute(sql).df()

q("""
SELECT COUNT(*) AS filas, COUNT(DISTINCT Invoice) AS facturas,
       COUNT(DISTINCT StockCode) AS skus, COUNT(DISTINCT Customer_ID) AS clientes,
       COUNT(DISTINCT Country) AS paises,
       MIN(InvoiceDate) AS desde, MAX(InvoiceDate) AS hasta
FROM r
""")

## 1. Perfil general de calidad

Antes de decidir nada, medir. Interesan cuatro familias de problema: valores
ausentes, valores imposibles, duplicación y códigos que no son producto.

In [ ]:
q("""
SELECT
    SUM(Description IS NULL)  AS sin_descripcion,
    SUM(Customer_ID IS NULL)  AS sin_cliente,
    SUM(Quantity < 0)         AS cantidad_negativa,
    SUM(Price = 0)            AS precio_cero,
    SUM(Price < 0)            AS precio_negativo,
    SUM(Invoice LIKE 'C%')    AS notas_credito,
    SUM(Quantity < 0 AND Invoice NOT LIKE 'C%') AS negativa_sin_nota
FROM r
""")

Dos lecturas importantes:

- **243.007 líneas sin `Customer_ID`** (22,8 %). Son ventas reales que no se pueden
  descartar, pero tampoco pueden entrar en un RFM: un análisis por cliente sobre
  filas sin cliente no significa nada.
- **3.457 líneas con cantidad negativa pero sin prefijo `C`**. No son devoluciones:
  son bajas de inventario (roturas, pérdidas) que el almacén registra como ventas
  negativas. Merecen una categoría propia.

## 2. El solape entre hojas

El libro trae una hoja por ejercicio. Comprobamos sus rangos reales antes de
concatenarlas — y ahí aparece el problema que más afecta a las cifras.

In [ ]:
q("""
SELECT source_sheet, MIN(InvoiceDate) AS desde, MAX(InvoiceDate) AS hasta, COUNT(*) AS filas
FROM r GROUP BY 1 ORDER BY 1
""")

Los rangos se pisan: la hoja 2009-2010 llega hasta el **9 de diciembre de 2010**
y la hoja 2010-2011 arranca el **1 de diciembre de 2010**. Nueve días están en las
dos. ¿Son las mismas filas o registros distintos?

In [ ]:
q("""
SELECT source_sheet, COUNT(*) AS filas, COUNT(DISTINCT Invoice) AS facturas,
       ROUND(SUM(Quantity * Price), 2) AS importe
FROM r
WHERE InvoiceDate >= '2010-12-01' AND InvoiceDate < '2010-12-10'
GROUP BY 1
""")

Idénticas: 22.523 filas, 1.088 facturas y £377.488 en cada hoja. Es duplicación
pura. Medimos cuánto distorsiona diciembre de 2010:

In [ ]:
q("""
WITH crudo AS (
    SELECT SUM(Quantity * Price) AS imp FROM r
    WHERE InvoiceDate >= '2010-12-01' AND InvoiceDate < '2011-01-01'),
limpio AS (
    SELECT SUM(Quantity * Price) AS imp FROM r
    WHERE InvoiceDate >= '2010-12-01' AND InvoiceDate < '2011-01-01'
      AND NOT (source_sheet = 'Year 2009-2010' AND InvoiceDate >= '2010-12-01'))
SELECT ROUND(crudo.imp, 2) AS con_duplicado,
       ROUND(limpio.imp, 2) AS sin_duplicado,
       ROUND(100.0 * (crudo.imp - limpio.imp) / limpio.imp, 1) AS inflacion_pct
FROM crudo, limpio
""")

**Un 50,4 % de facturación inflada** en ese mes si se concatena sin mirar.

→ *Regla 1:* recortar la hoja antigua en su frontera real y quedarse con la copia
de la hoja más reciente.

## 3. Duplicados exactos

Aparte del solape, quedan filas idénticas en los seis campos que definen una línea
de factura.

In [ ]:
q("""
SELECT Invoice, StockCode, Quantity, InvoiceDate, Price, Customer_ID, COUNT(*) AS veces
FROM r
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY veces DESC
LIMIT 5
""")

La factura 555524 repite la misma línea veinte veces, al mismo minuto y al mismo
precio. En un TPV eso es reescaneo: una venta real de veinte unidades se registra
como `Quantity = 20`, no como veinte filas de una unidad.

→ *Regla 2:* eliminar duplicados exactos. Cuesta £54.228 de facturación declarada,
y esa es exactamente la cifra que no era real.

## 4. Códigos que no son producto

El catálogo mezcla referencias con códigos de servicio contable.

In [ ]:
q("""
SELECT StockCode, ANY_VALUE(Description) AS descripcion, COUNT(*) AS lineas,
       ROUND(SUM(Quantity * Price), 0) AS importe
FROM r
WHERE StockCode NOT SIMILAR TO '[0-9]{5}[A-Za-z]*'
GROUP BY 1
ORDER BY ABS(SUM(Quantity * Price)) DESC
LIMIT 12
""")

`AMAZONFEE` suma **−£260.764** y `Adjust bad debt` **−£147.614**. Si entran como
ventas, cualquier KPI queda contaminado. Pero `POST` y `DOT` (portes) sí son ingreso
real, y códigos como `DCGS0058` son productos de verdad pese a no seguir el patrón
numérico.

→ *Regla 4:* no borrar, **etiquetar**. Cada línea recibe un `tipo_linea` (`producto`,
`envio`, `ajuste`, `voucher`, `prueba`) y cada consulta decide qué incluir.

## 5. Descripciones inconsistentes

Un mismo SKU puede arrastrar varios nombres.

In [ ]:
q("""
SELECT COUNT(*) AS skus_con_varias_descripciones FROM (
    SELECT StockCode FROM r WHERE Description IS NOT NULL
    GROUP BY 1 HAVING COUNT(DISTINCT UPPER(TRIM(Description))) > 1)
""")

In [ ]:
q(r"""
SELECT UPPER(TRIM(Description)) AS descripcion, COUNT(*) AS n
FROM r
WHERE Description ~ '(?i)(check|damage|lost|missing|found|smashed|broken|thrown)'
GROUP BY 1 ORDER BY n DESC LIMIT 10
""")

1.212 SKU afectados, y buena parte del ruido son anotaciones del almacén
(`check`, `found`, `missing`) coladas en el campo de descripción.

→ *Regla 3:* descripción canónica = la variante más frecuente de cada SKU,
excluyendo esas anotaciones.

## 6. Resumen de decisiones

| # | Regla | Motivo | Coste |
|---|---|---|---|
| 1 | Recortar el solape entre hojas | 9 días duplicados íntegros | −22.523 filas |
| 2 | Eliminar duplicados exactos | Reescaneo de línea en el TPV | −11.812 filas |
| 3 | Descripción canónica por SKU | 1.212 SKU con varios nombres | — |
| 4 | Etiquetar el tipo de línea | Ajustes contables contados como ventas | — |
| 5 | Cliente centinela `-1` | 22,8 % de líneas sin identificar | — |

Estas cinco reglas son las que implementa `sql/01_staging.sql`, con el mismo
razonamiento documentado en el propio SQL.